# Session 1 - Setup and Stimuli Design

Goal: design syntactic priming stimuli schema and reproducible generation flow.

This notebook is a playground. It intentionally avoids production implementation code.

## Learning Objectives

1. Understand syntactic priming experiment structure.
2. Build a stimulus schema that supports controlled comparisons.
3. Plan deterministic generation with random seeds.

In [ ]:
# Example only: imports and basic path setup
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_ROOT = PROJECT_ROOT / "src"
print("Project root:", PROJECT_ROOT)

## TODO Mapping to Source Files

Complete these modules after this notebook:
- src/stimuli/templates.py
- src/stimuli/generator.py
- src/stimuli/validator.py

## Session 1 Execution Spec (Important)

This section answers three practical questions: how to generate stimuli, how many to create, and where to store them.

1. How to generate?
- Default method: rule-based template expansion from src/stimuli/templates.py.
- Do not use free-form LLM generation as the main source of stimuli.
- LLM can be used only as an optional assistant to suggest candidates, then filter by your rules.

2. How many?
- Pilot demo: 40 to 80 stimuli total.
- Assignment target: 120 to 240 stimuli total.
- Better statistical power: 300 to 600 stimuli total.
- Keep counts balanced across conditions.

3. Where to store?
- Template/schema definitions: src/stimuli/templates.py
- Generated raw stimuli: data/raw/stimuli_session1_seed_<seed>.jsonl
- Validated/processed stimuli: data/processed/stimuli_balanced_seed_<seed>.jsonl

4. Minimal generation algorithm
- Step A: set condition list (for example: active_prime, passive_prime, dative variants).
- Step B: compute per-condition target counts using quotient/remainder allocation.
- Step C: expand templates and fill lexical slots (agent, patient, verb).
- Step D: assign stimulus_id, validate fields, then shuffle with a fixed seed.

## Session 1 Acceptance Checklist

By the end of Session 1, students should have:
1. A clear schema with required fields and metadata policy.
2. At least 3 to 5 syntactic conditions defined in templates.
3. A deterministic generator (same seed -> same output order).
4. Balanced condition counts in generated batch.
5. Exported JSONL files in data/raw and validated version in data/processed.

Minimum success criterion:
- Run a small batch (for example 20 to 40) and pass validation with zero critical errors.

## Common Student Mistakes and Fixes

1. Mistake: generating all records from one condition first, then stopping early.
Fix: allocate per-condition quotas before generation starts.

2. Mistake: hand-editing JSONL lines after generation.
Fix: update templates or generator logic, then regenerate.

3. Mistake: using free-form LLM output directly as final stimuli.
Fix: treat LLM output as candidate drafts; keep rule-based filtering mandatory.

4. Mistake: passing validation with warnings ignored.
Fix: define hard-fail and soft-fail checks explicitly and enforce the hard-fail gate.

In [ ]:
# Example only: balance and validation checks (toy demonstration)
from collections import Counter

toy_batch = [
    {"stimulus_id": "s0001", "condition": "active_prime", "prime_text": "...", "target_prompt": "...", "metadata": {"structure_label": "active"}},
    {"stimulus_id": "s0002", "condition": "passive_prime", "prime_text": "...", "target_prompt": "...", "metadata": {"structure_label": "passive"}},
    {"stimulus_id": "s0003", "condition": "active_prime", "prime_text": "...", "target_prompt": "...", "metadata": {"structure_label": "active"}},
    {"stimulus_id": "s0004", "condition": "passive_prime", "prime_text": "...", "target_prompt": "...", "metadata": {"structure_label": "passive"}},
]

required_fields = {"stimulus_id", "condition", "prime_text", "target_prompt", "metadata"}
allowed_conditions = {"active_prime", "passive_prime", "dative_double_object", "dative_prepositional"}

errors = []
seen_ids = set()
for i, rec in enumerate(toy_batch):
    missing = required_fields - set(rec.keys())
    if missing:
        errors.append(f"record[{i}] missing fields: {sorted(missing)}")
    sid = rec.get("stimulus_id")
    if sid in seen_ids:
        errors.append(f"record[{i}] duplicate stimulus_id: {sid}")
    seen_ids.add(sid)
    if rec.get("condition") not in allowed_conditions:
        errors.append(f"record[{i}] invalid condition: {rec.get('condition')}")

count_by_condition = Counter(rec["condition"] for rec in toy_batch)
if count_by_condition:
    spread = max(count_by_condition.values()) - min(count_by_condition.values())
    if spread > 1:
        errors.append(f"condition imbalance too high: spread={spread}")

print("Count by condition:", dict(count_by_condition))
print("Critical error count:", len(errors))
for msg in errors[:5]:
    print("-", msg)

## Validation Workflow (Step-by-Step)

Run validation in this order:
1. Record-level schema check:
- required fields exist
- data types are correct
- text fields are not empty

2. Batch-level integrity check:
- stimulus_id uniqueness
- allowed condition set only
- metadata consistency with condition

3. Balance check:
- summarize counts by condition
- verify max-min <= 1
- inspect lexical/topic distribution summaries

4. Export gate:
- only export JSONL when critical errors are zero.

Tip:
- Keep validator non-throwing; collect all errors into a list so students can fix in one pass.

## Detailed Balance Criteria

For this project, "balanced" means more than equal counts.

Hard balance rules (must pass):
1. Condition count balance: max(count) - min(count) <= 1.
2. All required conditions are present (no missing group).
3. No duplicated stimulus_id values across the batch.

Soft balance rules (recommended):
1. Lexical balance: agent/patient/verb frequencies are similar across conditions.
2. Length balance: average token length per condition is similar.
3. Topic balance: scenario topics are not concentrated in one condition.

Practical threshold suggestion:
- Hard rules: 100% pass required.
- Soft rules: keep deviation within about 10% for classroom projects.

## Function I/O Cheat Sheet (Session 1)

Use this section as an implementation contract. These are examples, not production logic.

| Function | Input (example) | Output (example) | Why this shape |
|---|---|---|---|
| `build_template_registry()` | None | `{\"active_prime_v1\": {...}, \"passive_prime_v1\": {...}}` | Single source of truth for templates and conditions |
| `render_template(template_name, variables)` | `\"active_prime_v1\"`, `{\"agent\": \"chef\", \"patient\": \"assistant\", \"verb_past\": \"praised\"}` | `\"The chef praised the assistant.\"` | Separates template structure from lexical slot filling |
| `list_template_names()` | None | `[\"active_prime_v1\", \"passive_prime_v1\"]` | Quick coverage checks in notebooks/tests |
| `generate_stimuli_batch(template_registry, batch_size, seed)` | registry, `80`, `42` | list of records with `stimulus_id/condition/prime_text/target_prompt/metadata` | One-step controlled generation at target scale |
| `randomize_stimuli_order(stimuli, seed)` | list of records, `42` | same records in deterministic shuffled order | Removes order bias while preserving reproducibility |
| `split_train_eval_stimuli(stimuli, eval_ratio, seed)` | records, `0.2`, `42` | `(train_list, eval_list)` | Clean separation for dev vs evaluation |
| `validate_stimulus_record(record)` | one record | `[]` or error messages list | Non-throwing validation supports classroom debugging |
| `validate_stimulus_batch(stimuli)` | records list | aggregated errors list | Catch systemic issues early |
| `summarize_condition_balance(stimuli)` | records list | `{\"active_prime\": 20, \"passive_prime\": 20}` | Balance is a core experimental requirement |

Implementation note:
- Students should manually design templates and schema.
- Students should not manually write 100+ JSONL lines one by one.
- Code should generate JSONL automatically from templates.

In [ ]:
# Toy I/O examples for Session 1 functions (for understanding only)

# 1) build_template_registry() expected shape example
template_registry_example = {
    "active_prime_v1": {
        "condition": "active_prime",
        "structure_label": "active",
        "prime_text_template": "The {agent} {verb_past} the {patient}.",
        "target_prompt_template": "Describe a scene involving a {agent} and a {patient}.",
        "lexical_constraints": {"agent": ["chef"], "patient": ["assistant"]}
    },
    "passive_prime_v1": {
        "condition": "passive_prime",
        "structure_label": "passive",
        "prime_text_template": "The {patient} was {verb_pp} by the {agent}.",
        "target_prompt_template": "Describe a scene involving a {agent} and a {patient}.",
        "lexical_constraints": {"agent": ["chef"], "patient": ["assistant"]}
    }
}

# 2) render_template(...) example result
render_input = {"template_name": "active_prime_v1", "variables": {"agent": "chef", "patient": "assistant", "verb_past": "praised"}}
render_output_example = "The chef praised the assistant."

# 3) generate_stimuli_batch(...) example output record shape
stimulus_record_example = {
    "stimulus_id": "s0001",
    "condition": "active_prime",
    "prime_text": "The chef praised the assistant.",
    "target_prompt": "Describe a scene involving a chef and an assistant.",
    "metadata": {"structure_label": "active", "template_name": "active_prime_v1"}
}

# 4) validator examples
valid_record_result_example = []
invalid_record_result_example = [
    "missing required field: target_prompt",
    "metadata.structure_label must be a non-empty string"
]

# 5) balance summary example
balance_summary_example = {"active_prime": 20, "passive_prime": 20}

print("Template keys:", list(template_registry_example.keys()))
print("Render output example:", render_output_example)
print("Record keys:", list(stimulus_record_example.keys()))
print("Balance summary:", balance_summary_example)

In [ ]:
# Example only: JSONL record format and auto-generation pattern (non-production)
import json
from pathlib import Path

# One record example
record = {
    "stimulus_id": "s0001",
    "condition": "active_prime",
    "prime_text": "The chef praised the assistant.",
    "target_prompt": "Describe a scene involving a chef and an assistant.",
    "metadata": {"structure_label": "active", "topic": "kitchen"}
}
print(json.dumps(record, ensure_ascii=False))

# Pseudo export path convention
output_path = Path("../data/raw/stimuli_session1_seed_42.jsonl")
print("Suggested output:", output_path)

# Your generator in src should create many records like this automatically.

## Scope and Quantity Guidance

Use these ranges as guidance, not hard rules:
- Smoke test: 12 to 20 stimuli (verify pipeline quickly).
- Pilot run: 40 to 80 stimuli (class assignment baseline).
- Stronger analysis preparation: 120 to 240 stimuli.

Balance rule: counts per condition should be as equal as possible.

Example with 4 conditions and 80 total stimuli:
- active_prime: 20
- passive_prime: 20
- dative_double_object: 20
- dative_prepositional: 20

## What Session 1 Is (and Is Not)

Session 1 theme: **experiment design + data schema + deterministic generator**, not large-scale manual labeling.

This session is **not** asking students to hand-write around 100 JSONL lines one by one.

Recommended workflow:
1. Manually design schema and templates (small, high-quality set).
2. Implement generator to expand templates automatically.
3. Validate and export JSONL programmatically.

Why this matters:
- Manual bulk writing is slow and error-prone.
- Template-driven generation is reproducible and easier to audit.
- Statistical comparisons later require balanced conditions, which is easier with code.

In [ ]:
# Example schema only (toy); you should improve this in src/common/types.py if needed
example_stimulus = {
    "stimulus_id": "s001",
    "condition": "active_prime",
    "prime_text": "The chef praised the assistant.",
    "target_prompt": "Describe a scene involving a chef and an assistant.",
    "metadata": {"structure_label": "active"}
}
example_stimulus

## Student Tasks

1. Implement template registry functions in src/stimuli/templates.py.
2. Implement balanced batch generation in src/stimuli/generator.py.
3. Implement schema checks in src/stimuli/validator.py.
4. Keep generation deterministic using `seed`.

In [ ]:
# Playground call pattern (will fail until TODOs are implemented)
from src.stimuli.templates import build_template_registry
from src.stimuli.generator import generate_stimuli_batch
from src.stimuli.validator import validate_stimulus_batch

registry = build_template_registry()
batch = generate_stimuli_batch(registry, batch_size=20, seed=42)
errors = validate_stimulus_batch(batch)
print("stimuli:", len(batch), "errors:", len(errors))

## Agentic Preview (Minimal Loop in Session 1)


To keep the project clearly agent-oriented, we add a lightweight agentic loop preview here:
1. Experimenter side prepares one prompt package from one stimulus record.
2. Subject side receives it and returns one response.
3. We log a trial artifact with minimal metadata.

Important boundary:
- This is a teaching preview, not production orchestration.
- Full multi-agent batch pipeline is implemented in Session 2.

Why this helps in Session 1:
- Students see how stimuli design directly feeds agent behavior evaluation.
- Students understand why strict schema and validation are required for agent loops.

In [ ]:
# Minimal agentic loop example (toy, non-production)
from dataclasses import dataclass

@dataclass
class ExperimenterAgentMock:
    name: str = "experimenter"

    def prepare_prompt_package(self, stimulus: dict) -> dict:
        return {
            "role": self.name,
            "condition": stimulus["condition"],
            "prime_text": stimulus["prime_text"],
            "target_prompt": stimulus["target_prompt"],
            "metadata": stimulus.get("metadata", {}),
        }

@dataclass
class SubjectAgentMock:
    name: str = "subject_llm"

    def respond(self, prompt_package: dict) -> dict:
        # Placeholder response to demonstrate interface only.
        return {
            "role": self.name,
            "response_text": "The assistant was praised by the chef.",
            "metadata": {"note": "toy response for loop demonstration"},
        }

stimulus = {
    "stimulus_id": "s0001",
    "condition": "active_prime",
    "prime_text": "The chef praised the assistant.",
    "target_prompt": "Describe a scene involving a chef and an assistant.",
    "metadata": {"structure_label": "active", "template_name": "active_prime_v1"},
}

experimenter = ExperimenterAgentMock()
subject = SubjectAgentMock()
prompt_package = experimenter.prepare_prompt_package(stimulus)
subject_response = subject.respond(prompt_package)

trial_artifact = {
    "stimulus_id": stimulus["stimulus_id"],
    "condition": stimulus["condition"],
    "prompt_package": prompt_package,
    "subject_response": subject_response,
}

print("Agentic trial artifact keys:", list(trial_artifact.keys()))
print("Condition:", trial_artifact["condition"])
print("Response:", trial_artifact["subject_response"]["response_text"])